In [1]:
#load packages
import os
import numpy as np
import pandas as pd


In [2]:
#import daily temp data

file_path = "/group/moniergrp/dbaral/run_project/input_data/gridmet"

df = pd.read_csv(file_path + "/Daily_Rice_Temperature_1979_2023_v2.csv")
df = df.sort_values(by = ['county', 'year', 'month', 'day'])

print(df.head())
print(df.shape)

      county        date  year  month  day       tmmn       tmmx      tmean
17108  Butte  1979-05-12  1979      5   12  14.835787  32.610837  23.723312
17117  Butte  1979-05-13  1979      5   13  17.088482  33.863273  25.475878
17126  Butte  1979-05-14  1979      5   14  17.851497  33.443441  25.647469
17135  Butte  1979-05-15  1979      5   15  13.445309  30.332816  21.889063
17144  Butte  1979-05-16  1979      5   16  11.304421  29.835681  20.570051
(59985, 8)


In [3]:
#Checking for complete data in df

expected_years = range(1979, 2024)
expected_months = range(4, 11)
expected_days = range(1, 32)

counties = df['county'].unique()
years = df['year'].unique()
months = df['month'].unique()
days = df['day'].unique()

print("Checking data coverage:")
print(f"Expected_years: {list(expected_years)}")
print(f"Years in data: {years}")
print(f"Expected_months: {list(expected_months)}")
print(f"Months in data: {list(months)}")
print(f"Expected_days: {list(expected_days)}")
print(f"Days in data: {days}")

#Checking if each county has all years, months and days and if there are any missing values

for county in counties:
    county_data = df[df['county'] == county]
    county_years = county_data['year'].unique()
    county_months = county_data['month'].unique()
    county_days = county_data['day'].unique()
    print(f"\nCounty: {county}")
    print(f"Years in data: {county_years}")
    print(f"Months in data: {county_months}")
    print(f"Days in data: {county_days}")

    missing_values = county_data.isnull().sum()
    if missing_values.sum() > 0:
        print(f"\nMissing values for {county}:")
        print(missing_values[missing_values > 0])
    else:
        print(f"\nNo missing values for {county}")



Checking data coverage:
Expected_years: [1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Years in data: [1979 1980 1981 1982 1983 1984 1985 1986 1987 1988 1989 1990 1991 1992
 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005 2006
 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020
 2021 2022 2023]
Expected_months: [4, 5, 6, 7, 8, 9, 10]
Months in data: [np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Expected_days: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Days in data: [12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31  1  2  3  4
  5  6  7  8  9 10 11]

County: Butte
Years in data: [1979 1980 1981 1982 1983 1984 1985 1

From poster by Bruce:

472 GDD for Planting(PL) to Panicle Initiation(PI)

796 GDD for Planting to Heading(HD)

1007 GDD for Planting to R7


From: Sharifi et al 2018 _ Air and Water temp research

PL-PI: 454 GDD

PI_HD: 356 GDD

HD_R7: 203 GDD

HD + 30 days == grain fill period






In [4]:
# base temp, lower temp, optimum temp and threshold gdd for each rice stage: needed to calculated daily gdd
#temp are in C
#referenced from Sharifi et al 2016 - for medium grain rice
# Pl_PI: Planting(PL) to Panicle Initiation(PI)
#PI_HD: Panicle Initiation (PI) to Heading (HD)
#HD_R7: Heading(HD) to R7 stage

growth_stages = {
    'PL_PI' : {
       'tbase': 10,
       'tlower': 14.30,
       'topt': 27.73,
       'threshold': 454
   },
   'PI_HD' : {
       'tbase': 9.8,
       'tlower': 15.37,
       'topt': 30.73,
       'threshold': 356
   },
   'HD_R7' : {
       'tbase': 8.9,
       'tlower': 15.93,
       'topt': 33.03,
       'threshold': 203
   }
}

gdd formula

TTt = max(0, [0.5(Tmax + Tmin)] - Tbase)
Tmin = Tl if Tmin > Tl;
Tmax = Topt if Tmax > T opt

where TTt is the thermal time at time t, Tmax is the daily maximum temperature, Tmin is the daily minimum temperature, Tb is the base temperature, Tl is the lower threshold, and Topt is the optimum threshold. There is no development (i.e., TTt below zero is set to zero) below Tb (here 10°C), and since Tmax that are greater than Topt are set equal to Topt, there is no increase in development for daily maximum temperatures above Topt. Similarly, for Tmin, since Tmin that are greater than Tl are set equal to Tl, there is no increase in development for daily minimum temperatures above Tl.


In [5]:
#function to calculate daily gdd
def calculate_daily_gdd (tmmn, tmmx, tbase, tlower, topt):
  tmmx_adj = min(tmmx, topt)
  tmmn_adj = min(tmmn, tlower)
  tmean_adj = (tmmx_adj + tmmn_adj)/2
  gdd = max(tmean_adj - tbase, 0)
  return gdd

In [6]:
# Test cases for calculate_daily_gdd function

# Test case 1: Tmin and Tmax within thresholds
tmmn_test1, tmmx_test1, tbase_test1, tlower_test1, topt_test1 = 10, 30, 10, 14, 20
gdd_test1 = calculate_daily_gdd(tmmn_test1, tmmx_test1, tbase_test1, tlower_test1, topt_test1)
print(f"Test Case 1: tmmn={tmmn_test1}, tmmx={tmmx_test1}, tbase={tbase_test1}, tlower={tlower_test1}, topt={topt_test1} -> GDD = {gdd_test1}")

# Test case 2: Tmin below tbase
tmmn_test2, tmmx_test2, tbase_test2, tlower_test2, topt_test2 = 5, 20, 10, 14, 30
gdd_test2 = calculate_daily_gdd(tmmn_test2, tmmx_test2, tbase_test2, tlower_test2, topt_test2)
print(f"Test Case 2: tmmn={tmmn_test2}, tmmx={tmmx_test2}, tbase={tbase_test2}, tlower={tlower_test2}, topt={topt_test2} -> GDD = {gdd_test2}")

# Test case 3: Tmax above topt
tmmn_test3, tmmx_test3, tbase_test3, tlower_test3, topt_test3 = 15, 35, 10, 14, 30
gdd_test3 = calculate_daily_gdd(tmmn_test3, tmmx_test3, tbase_test3, tlower_test3, topt_test3)
print(f"Test Case 3: tmmn={tmmn_test3}, tmmx={tmmx_test3}, tbase={tbase_test3}, tlower={tlower_test3}, topt={topt_test3} -> GDD = {gdd_test3}")

# Test case 4: Tmin above tlower
tmmn_test4, tmmx_test4, tbase_test4, tlower_test4, topt_test4 = 18, 25, 10, 14, 30
gdd_test4 = calculate_daily_gdd(tmmn_test4, tmmx_test4, tbase_test4, tlower_test4, topt_test4)
print(f"Test Case 4: tmmn={tmmn_test4}, tmmx={tmmx_test4}, tbase={tbase_test4}, tlower={tlower_test4}, topt={topt_test4} -> GDD = {gdd_test4}")

# Test case 5: Both Tmin and Tmax outside thresholds
tmmn_test5, tmmx_test5, tbase_test5, tlower_test5, topt_test5 = 8, 40, 10, 14, 30
gdd_test5 = calculate_daily_gdd(tmmn_test5, tmmx_test5, tbase_test5, tlower_test5, topt_test5)
print(f"Test Case 5: tmmn={tmmn_test5}, tmmx={tmmx_test5}, tbase={tbase_test5}, tlower={tlower_test5}, topt={topt_test5} -> GDD = {gdd_test5}")

# Test case 6: Tmean below tbase
tmmn_test6, tmmx_test6, tbase_test6, tlower_test6, topt_test6 = 5, 10, 10, 14, 30
gdd_test6 = calculate_daily_gdd(tmmn_test6, tmmx_test6, tbase_test6, tlower_test6, topt_test6)
print(f"Test Case 6: tmmn={tmmn_test6}, tmmx={tmmx_test6}, tbase={tbase_test6}, tlower={tlower_test6}, topt={topt_test6} -> GDD = {gdd_test6}")

Test Case 1: tmmn=10, tmmx=30, tbase=10, tlower=14, topt=20 -> GDD = 5.0
Test Case 2: tmmn=5, tmmx=20, tbase=10, tlower=14, topt=30 -> GDD = 2.5
Test Case 3: tmmn=15, tmmx=35, tbase=10, tlower=14, topt=30 -> GDD = 12.0
Test Case 4: tmmn=18, tmmx=25, tbase=10, tlower=14, topt=30 -> GDD = 9.5
Test Case 5: tmmn=8, tmmx=40, tbase=10, tlower=14, topt=30 -> GDD = 9.0
Test Case 6: tmmn=5, tmmx=10, tbase=10, tlower=14, topt=30 -> GDD = 0


In [7]:
#Initialize results storage

results = []   #Remember this has to be outside the loop and should be done before starting the for loop

#process each county and year combination

for (county, year), group in df.groupby(['county', 'year']):
  season_data = group.copy()
  season_data = season_data.sort_values(by= 'date')

  #Convert 'date' column to datetime objects for reliable comparison
  season_data['date'] = pd.to_datetime(season_data['date'])

  # Use the first date in the 'date' column as planting date
  planting_date = season_data['date'].iloc[0] if not season_data.empty else None
  # Use the last date in the 'date' column as harvest date
  harvest_date = season_data['date'].iloc[-1] if not season_data.empty else None


  # Filtering data from the planting date onwards
  season_data = season_data[(season_data['date'] >= planting_date)].copy() # Use .copy() to avoid SettingWithCopyWarning

  #print(season_data.head())
  #print(season_data.info)
  #print(season_data.describe())

  #Initialize growth tracking
  current_stage = 'PL_PI'
  stage_gdd = 0
  accumulated_gdd = 0
  transition_dates = {
      'pl_date': season_data.iloc[0]['date'] if not season_data.empty else None, # To handle empty season_data
      'pi_date': None,
      'hd_date': None,
      'r7_date': None,
      'grain_fill_start_date': None,
      'grain_fill_end_date': None
  }

  # Handling case where season_data might be empty after filtering
  if season_data.empty:
    print(f"No data for growing season in {county}, {year}")
    continue # Skip to the next group if no data for the season


  daily_results = []

  for index, row in season_data.iterrows():
      # Calculate daily GDD based on the current stage's parameters
      stage_params = growth_stages.get(current_stage)
      daily_gdd = 0

      if stage_params:
          daily_gdd = calculate_daily_gdd(
              row['tmmn'],
              row['tmmx'],
              stage_params['tbase'],
              stage_params['tlower'],
              stage_params['topt']
          )
          stage_gdd += daily_gdd
          accumulated_gdd += daily_gdd


      # Check for stage transition based on GDD
      new_stage = None
      if current_stage in growth_stages:
          stage_params = growth_stages.get(current_stage)
          if stage_params and stage_gdd >= stage_params['threshold']:
              if current_stage == 'PL_PI':
                  transition_dates['pi_date'] = row['date']
                  new_stage = 'PI_HD'
              elif current_stage == 'PI_HD':
                  transition_dates['hd_date'] = row['date']
                  new_stage = 'HD_R7'
              elif current_stage == 'HD_R7':
                  transition_dates['r7_date'] = row['date']
                  new_stage = 'completed' # Transition to a completed stage

      # Update current stage and reset stage_gdd if a transition occurred
      if new_stage:
          current_stage = new_stage
          stage_gdd = 0 # Reset stage_gdd for the new stage

      # Check for grain fill start date based on hd_date + 0 days
      if current_stage == 'completed' and transition_dates['hd_date'] is not None and transition_dates['grain_fill_start_date'] is None:
          transition_dates['grain_fill_start_date'] = transition_dates['hd_date']
          transition_dates['grain_fill_end_date'] = transition_dates['grain_fill_start_date'] + pd.Timedelta(days=30)


      # If in the grain fill period, update the stage
      if transition_dates['grain_fill_start_date'] is not None and transition_dates['grain_fill_end_date'] is not None:
          if row['date'] >= transition_dates['grain_fill_start_date'] and row['date'] <= transition_dates['grain_fill_end_date']:
              current_stage = 'grain_fill'

      # Only append results if the date is within the extended season
      if row['date'] <= harvest_date:
          # Add to overall results
          daily_results.append({
              'county': county,
              'date': row['date'],
              'year': year,
              'month': row['month'],
              'day': row['day'],
              'stage': current_stage, # Use current_stage after transition check
              'daily_gdd': daily_gdd,
              'accumulated_gdd': accumulated_gdd,
              'stage_gdd': stage_gdd, # stage_gdd is reset upon transition
              'pl_date': transition_dates['pl_date'],
              'pi_date': transition_dates['pi_date'],
              'hd_date': transition_dates['hd_date'],
              'r7_date': transition_dates['r7_date'],
              'grain_fill_start_date': transition_dates['grain_fill_start_date'],
              'grain_fill_end_date': transition_dates['grain_fill_end_date']
          })
  results.extend(daily_results)

In [8]:
results_df = pd.DataFrame(results)
print(results_df.head())
print(results_df.shape)

  county       date  year  month  day  stage  daily_gdd  accumulated_gdd  \
0  Butte 1979-05-12  1979      5   12  PL_PI  11.015000        11.015000   
1  Butte 1979-05-13  1979      5   13  PL_PI  11.015000        22.030000   
2  Butte 1979-05-14  1979      5   14  PL_PI  11.015000        33.045000   
3  Butte 1979-05-15  1979      5   15  PL_PI  10.587655        43.632655   
4  Butte 1979-05-16  1979      5   16  PL_PI   9.517211        53.149865   

   stage_gdd    pl_date pi_date hd_date r7_date grain_fill_start_date  \
0  11.015000 1979-05-12     NaT     NaT     NaT                   NaT   
1  22.030000 1979-05-12     NaT     NaT     NaT                   NaT   
2  33.045000 1979-05-12     NaT     NaT     NaT                   NaT   
3  43.632655 1979-05-12     NaT     NaT     NaT                   NaT   
4  53.149865 1979-05-12     NaT     NaT     NaT                   NaT   

  grain_fill_end_date  
0                 NaT  
1                 NaT  
2                 NaT  
3       

In [9]:
print(f"Results exported for {len(df['county'].unique())} counties and {len(df['year'].unique())} years.")


Results exported for 9 counties and 45 years.


In [10]:
#Converting date cols to datetime format

date_cols = ['pl_date', 'pi_date', 'hd_date', 'r7_date', 'grain_fill_start_date', 'grain_fill_end_date']
results_df[date_cols] = results_df[date_cols].apply(pd.to_datetime)
# print(results_df.dtypes)

#calculating day of the year for each of the date cols
for col in date_cols:
    results_df[f'{col}_doy'] = results_df[col].dt.dayofyear
# print(results_df.head)

#converting date column to datetime format
results_df['date'] = pd.to_datetime(results_df['date'])
print(results_df.dtypes)

# Calculate days after planting for pi_date and hd_date
results_df['pi_dap'] = (results_df['pi_date'] - results_df['pl_date']).dt.days
results_df['hd_dap'] = (results_df['hd_date'] - results_df['pl_date']).dt.days
results_df['r7_dap'] = (results_df['r7_date'] - results_df['pl_date']).dt.days
results_df['grain_fill_dap'] = (results_df['grain_fill_start_date'] - results_df['pl_date']).dt.days
print(results_df.head)

results_df.to_csv(file_path + '/Rice_Growth_Stages_GDD_Results_1979_2023_v2.csv', index=False)

county                               object
date                         datetime64[ns]
year                                  int64
month                                 int64
day                                   int64
stage                                object
daily_gdd                           float64
accumulated_gdd                     float64
stage_gdd                           float64
pl_date                      datetime64[ns]
pi_date                      datetime64[ns]
hd_date                      datetime64[ns]
r7_date                      datetime64[ns]
grain_fill_start_date        datetime64[ns]
grain_fill_end_date          datetime64[ns]
pl_date_doy                           int32
pi_date_doy                         float64
hd_date_doy                         float64
r7_date_doy                         float64
grain_fill_start_date_doy           float64
grain_fill_end_date_doy             float64
dtype: object
<bound method NDFrame.head of       county       date  year  m

In [11]:
#creating new data frame to extract transition dates only

transition_dates = results_df.groupby(['county', 'year']).agg({
    'pl_date': 'first',
    'pi_date': 'first',
    'hd_date': 'first',
    'r7_date': 'first',
    'grain_fill_start_date': 'first',
    'grain_fill_end_date': 'first',
    'pl_date_doy': 'first',
    'pi_date_doy': 'first',
    'hd_date_doy': 'first',
    'r7_date_doy': 'first',
    'grain_fill_start_date_doy': 'first',
    'grain_fill_end_date_doy': 'first',
    'pi_dap': 'first',
    'hd_dap': 'first',
    'r7_dap': 'first',
    'grain_fill_dap': 'first'
}).reset_index()

transition_dates.to_csv(file_path + '/Phenology_Transition_1979_2023_v2.csv', index=False)

In [12]:
#merge temp and gdd data to work for stress indices because we can identify the stages: Booting and Flowering only from gdd
#and to calulate stres indices we need daily temp data

temp_df = df
gdd_df = results_df

#changing date column to pd datetime format for temp data
temp_df['date'] = pd.to_datetime(temp_df['date'])

temp_gdd_df = pd.merge(
    left = temp_df,
    right = gdd_df,
    on =  ['county', 'year', 'month', 'day', 'date'],
    how = 'right'
)

temp_gdd_df.to_csv(file_path +'/Combined_Temp_GDD_1979_2024_v2.csv', index=False)


In [13]:
#Defining booting and flowering stage
##Booting is 7 days after Panicle Initiation and until 7 days before 50% heading
##Flowering is 7 days before 50% heading and 7 days after 50% heading
##Grainfill period is 30 days following 50%heading


In [14]:
results = []

grouped = temp_gdd_df.groupby(['county', 'year'])

#Loop through each group
for i, ((county, year), group) in enumerate(grouped):
    group = group.copy() # Get the group DataFrame explicitly
    #print(f"Group {i}: {county}, {year}, {len(group)} rows")
    #print(group.dtypes)
    #get first dates for this group
    first_pi_date = group['pi_date'].dropna().iloc[0]
    first_hd_date = group['hd_date'].dropna().iloc[0]
    #print(first_pi_date)
    #print(first_hd_date)

    booting_start = first_pi_date + pd.Timedelta(days = 7)
    booting_end = first_hd_date - pd.Timedelta(days = 7)

    flowering_start = first_hd_date - pd.Timedelta(days = 7)
    flowering_end = first_hd_date + pd.Timedelta(days = 7)

    grain_fill_start = first_hd_date
    grain_fill_end = first_hd_date + pd.Timedelta(days = 30)

    # Initialize the 'stage' column with a default value

    group['stage'] = 'growth stage' # Default stage

    #filtering for booting stage and naming the stage as booting

    group.loc[
        (group['date'] >= booting_start) &
        (group['date'] <= booting_end),
        'stage'
    ] = 'booting'

    #filtering for flowering stage and naming the stage column as flowering
    flowering_mask = (
        (group['date'] >= flowering_start) &
        (group['date'] <= first_hd_date)
    )

    group.loc[flowering_mask, 'stage'] = 'flowering'

    #filtering for overlapping period of flowering and grain fill stage and naming it flowering and grainfill

    flowering_grainfill_mask = (
        (group["date"] > first_hd_date) &
        (group["date"] <= flowering_end)
    )

    group.loc[flowering_grainfill_mask, 'stage'] = 'flowering_grainfill'


    #filtering for grain_fil period and naming it as grainfill

    grain_fill_mask = (
        (group['date'] > flowering_end) &
        (group['date'] <= grain_fill_end)
    )

    group.loc[grain_fill_mask, 'stage'] = 'grain_fill'


    # Debugging print statements for flowering mask
    if i == 0:
      print(f"--- Debugging Stage Assignment for {county}, {year} ---")
      print(f"Booting start date: {booting_start}, Booting end date: {booting_end}")
      print(f"Flowering start date: {flowering_start}, Flowering end date: {flowering_end}")
      print(f"Dates marked as flowering: {group.loc[flowering_mask, 'date']}")
      print(f"Dates marked as flowering and grainfill: {group.loc[flowering_grainfill_mask, 'date']}")
      print(f"Grain fill start date: {grain_fill_start}, Grain fill end date: {grain_fill_end}")
      print(f"Dates marked as grain filling: {group.loc[grain_fill_mask, 'date']}")

      print("--- End Debugging ---")


    # Add tmmx_gf and tmmn_gf columns for grain fill period
    group['tmmx_gf'] = np.nan
    group['tmmn_gf'] = np.nan

    # Calculate max and min temperature for the entire grain fill period for this county and year
    grain_fill_period_data = group[group['stage'] == 'grain_fill']
    if not grain_fill_period_data.empty:
        max_tmmx_gf = grain_fill_period_data['tmmx'].max()
        min_tmmn_gf = grain_fill_period_data['tmmn'].min()
        # Assign the calculated max and min to all rows within the grain fill period for this group
        group.loc[group['stage'] == 'grain_fill', 'tmmx_gf'] = max_tmmx_gf
        group.loc[group['stage'] == 'grain_fill', 'tmmn_gf'] = min_tmmn_gf

    results.append(group)

#combine all groups
final_results = pd.concat(results)

--- Debugging Stage Assignment for Butte, 1979 ---
Booting start date: 1979-07-01 00:00:00, Booting end date: 1979-07-17 00:00:00
Flowering start date: 1979-07-17 00:00:00, Flowering end date: 1979-07-31 00:00:00
Dates marked as flowering: 66   1979-07-17
67   1979-07-18
68   1979-07-19
69   1979-07-20
70   1979-07-21
71   1979-07-22
72   1979-07-23
73   1979-07-24
Name: date, dtype: datetime64[ns]
Dates marked as flowering and grainfill: 74   1979-07-25
75   1979-07-26
76   1979-07-27
77   1979-07-28
78   1979-07-29
79   1979-07-30
80   1979-07-31
Name: date, dtype: datetime64[ns]
Grain fill start date: 1979-07-24 00:00:00, Grain fill end date: 1979-08-23 00:00:00
Dates marked as grain filling: 81    1979-08-01
82    1979-08-02
83    1979-08-03
84    1979-08-04
85    1979-08-05
86    1979-08-06
87    1979-08-07
88    1979-08-08
89    1979-08-09
90    1979-08-10
91    1979-08-11
92    1979-08-12
93    1979-08-13
94    1979-08-14
95    1979-08-15
96    1979-08-16
97    1979-08-17
98    

Calculating Stress Indices:

*Theta is the Temp Threshold
*Ti is min temp for cold stress, and Ti is max temp for heat stress
  
Heat Stress = Ti - Theta if Ti >= theta;
            = 0, if Ti < Theta

Cold Stress = Theta - Ti if Ti <= theta;
            = 0, if Ti > theta

In [15]:

#Extract data frame for booting, flowering and HD_R7 stage only, AND all rows where grain_fill is True

booting_flowering_grainfill_df = final_results[
    final_results['stage'].isin(['booting', 'flowering', 'flowering_grainfill', 'HD_R7', 'grain_fill'])
].copy()

booting_flowering_grainfill_df.to_csv(file_path + '/Booting_Flowering_GrainFill_1979_2023_v2.csv', index=False)

In [16]:
# creating stage table


In [17]:
# INPUT / OUTPUT
in_csv  = "/Booting_Flowering_GrainFill_1979_2023_v2.csv"
out_csv = "/9_County_Stage_Dates_1979_2023.csv"

# How to build the 3 output stages from the daily "stage" labels in the big file
STAGE_MAP = {
    "booting":    ["booting"],
    "flowering":  ["flowering", "flowering_grainfill"],      # include overlap
    "grain_fill": ["grain_fill", "flowering_grainfill"],     # include overlap
}

df = pd.read_csv(file_path + in_csv)

# make sure date is datetime
df["date"] = pd.to_datetime(df["date"])

rows = []
for (county, year), g in df.groupby(["county", "year"], sort=True):
    for out_stage, labels in STAGE_MAP.items():
        sub = g[g["stage"].isin(labels)]
        if sub.empty:
            continue
        rows.append({
            "county": county,
            "year": int(year),
            "stage": out_stage,
            "start_date": sub["date"].min().date().isoformat(),
            "end_date": sub["date"].max().date().isoformat(),
        })

stage_dates = pd.DataFrame(rows).sort_values(["county", "year", "stage"]).reset_index(drop=True)
stage_dates.to_csv(file_path + out_csv, index=False)

print("Saved:", out_csv)
print(stage_dates.head(10))

Saved: /9_County_Stage_Dates_1979_2023.csv
  county  year       stage  start_date    end_date
0  Butte  1979     booting  1979-07-01  1979-07-16
1  Butte  1979   flowering  1979-07-17  1979-07-31
2  Butte  1979  grain_fill  1979-07-25  1979-08-23
3  Butte  1980     booting  1980-07-10  1980-07-25
4  Butte  1980   flowering  1980-07-26  1980-08-09
5  Butte  1980  grain_fill  1980-08-03  1980-09-01
6  Butte  1981     booting  1981-07-03  1981-07-17
7  Butte  1981   flowering  1981-07-18  1981-08-01
8  Butte  1981  grain_fill  1981-07-26  1981-08-24
9  Butte  1982     booting  1982-07-03  1982-07-18
